# Synthetic Data Generation for Tool-Calling Fine-Tuning

Fine-tune `gpt-4.1-mini` to call a catalog of tools correctly, using training data generated by the Foundry Data Generation API from a static OpenAPI tool spec.

End-to-end:
1. Convert an OpenAI-format tool catalog to OpenAPI 3.0 and upload it
2. Generate examples with the `ToolUseFineTuning` recipe
3. Split the examples into train / validation / test sets
4. Evaluate the base deployment with structural tool-call matching
5. Submit and monitor a fine-tuning job
6. Validate an existing project deployment of the fine-tuned model
7. Evaluate that deployment on the same held-out set and report lift

The notebook uses `AIProjectClient` with a Foundry project endpoint and `DefaultAzureCredential`. Data generation uses the SDK poller, evaluation uses the stable `azure-ai-evaluation` runner, and model/files/fine-tuning calls use only the documented client returned by `AIProjectClient.get_openai_client()`. No API keys, account endpoints, direct OpenAI client construction, or raw management URLs are used.

## 1. Setup & Configuration

In [ ]:
import os, json, time, random
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
TEACHER_MODEL = os.environ.get("AZURE_TEACHER_DEPLOYMENT", "gpt-4.1")
STUDENT_MODEL = os.environ.get("AZURE_STUDENT_DEPLOYMENT", "gpt-4.1-mini")
TRAINING_TYPE = os.environ.get("AZURE_FINE_TUNING_TRAINING_TYPE", "GlobalStandard")
DEPLOY_NAME = os.environ.get("AZURE_FINE_TUNED_DEPLOYMENT", "zava-tools-ft-demo")
RESUME_FT_JOB_ID = os.environ.get("AZURE_FINE_TUNING_JOB_ID")
RESUME_DATAGEN_JOB_IDS = [
    value for value in os.environ.get("AZURE_DATAGEN_JOB_IDS", "").split(",") if value
]

WORK = Path("./run").resolve()
WORK.mkdir(exist_ok=True)

credential = DefaultAzureCredential()
project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
    polling_interval=15,
)
model_client = project.get_openai_client()
RUN_STATE = {
    "project_endpoint": PROJECT_ENDPOINT,
    "auth": "DefaultAzureCredential",
    "teacher_model": TEACHER_MODEL,
    "student_model": STUDENT_MODEL,
    "training_type": TRAINING_TYPE,
    "fine_tuned_deployment": DEPLOY_NAME,
    "sdk_usage": {
        "project": "AIProjectClient",
        "model_and_fine_tuning": "AIProjectClient.get_openai_client",
        "data_generation": "AIProjectClient.beta.datasets poller",
        "evaluation": "azure.ai.evaluation.evaluate",
    },
    "stages": {},
}

print(f"Project:    {PROJECT_ENDPOINT}")
print(f"Teacher:    {TEACHER_MODEL}")
print(f"Student:    {STUDENT_MODEL}")
print(f"Training:   {TRAINING_TYPE}")
print(f"Deployment: {DEPLOY_NAME}")
print(f"Work dir:   {WORK}")

## 2. Inline helpers: tool catalog conversion + Foundry data generation

The conversion helper maps the OpenAI-format tool array to the OpenAPI 3.0 input required by `ToolUseFineTuning`. Data generation uses the SDK's documented `DatasetGenerationLROPoller`; the data-generation operation is still under `project.beta.datasets` in `azure-ai-projects` 2.x.

In [ ]:
def openai_tools_to_openapi(openai_tools, title="Tools"):
    """Convert an OpenAI chat-completions tools array to an OpenAPI 3.0 spec.

    The Foundry ToolUseFineTuning recipe consumes OpenAPI 3.0, not the
    OpenAI tools-array. Each tool function becomes a POST operation under
    /<tool_name>.
    """
    def clean_schema(value):
        if isinstance(value, dict):
            cleaned = {key: clean_schema(item) for key, item in value.items()}
            if cleaned.get("required") == []:
                cleaned.pop("required")
            return cleaned
        if isinstance(value, list):
            return [clean_schema(item) for item in value]
        return value

    paths = {}
    for t in openai_tools:
        fn = t.get("function", {})
        name = fn["name"]
        params = clean_schema(fn.get("parameters") or {"type": "object", "properties": {}})
        paths[f"/{name}"] = {
            "post": {
                "operationId": name,
                "summary": fn.get("description", ""),
                "requestBody": {
                    "required": True,
                    "content": {"application/json": {"schema": params}},
                },
                "responses": {"200": {"description": "Success"}},
            }
        }
    return {
        "openapi": "3.0.3",
        "info": {"title": title, "version": "1.0.0"},
        "paths": paths,
    }



def submit_foundry_datagen(*, file_id, teacher, max_samples, output_name, timeout_s=1800):
    """Submit a ToolUseFineTuning job with the SDK poller and download its JSONL output."""
    from azure.ai.projects.models import (
        DataGenerationJob,
        DataGenerationJobInputs,
        DataGenerationJobScenario,
        DataGenerationJobOutputOptions,
        ToolUseFineTuningDataGenerationJobOptions,
        FileDataGenerationJobSource,
        DataGenerationModelOptions,
    )

    options = ToolUseFineTuningDataGenerationJobOptions(
        max_samples=max_samples,
        model_options=DataGenerationModelOptions(model=teacher),
    )
    job = DataGenerationJob(inputs=DataGenerationJobInputs(
        name=output_name,
        scenario=DataGenerationJobScenario.SUPERVISED_FINETUNING,
        sources=[FileDataGenerationJobSource(id=file_id)],
        options=options,
        output_options=DataGenerationJobOutputOptions(name=output_name),
    ))

    print(f"Submitting data-generation job (max_samples={max_samples})...")
    poller = project.beta.datasets.begin_create_generation_job(job=job)
    job_id = poller.details.get("job_id")
    if not job_id:
        raise RuntimeError("Data-generation submission did not return a job ID")
    RUN_STATE.setdefault("data_generation_job_ids", []).append(job_id)
    print(f"  job.id={job_id}")

    result = poller.result(timeout=timeout_s)
    outputs = result.outputs or []
    if not outputs:
        raise RuntimeError(f"Data-generation job {job_id} completed without output files")

    out_path = WORK / f"{output_name}_dg.jsonl"
    with out_path.open("wb") as destination:
        for output in outputs:
            destination.write(model_client.files.content(file_id=output.id).read())
    RUN_STATE.setdefault("data_generation_statuses", []).append(poller.status())
    print(f"  generated {result.generated_samples} sample(s) in {len(outputs)} file(s)")
    print(f"  saved {out_path.name} ({out_path.stat().st_size:,} bytes)")
    return out_path

## 3. Convert + upload the Zava tool catalog

The bundled fixture contains 15 retail-support tools. The notebook converts it to OpenAPI 3.0 and uploads it through the model/files client obtained from the project client.

In [ ]:
OPENAI_TOOLS = json.loads(Path("fixtures/zava_tools_openai.json").read_text(encoding="utf-8"))
print(f"Loaded {len(OPENAI_TOOLS)} tools: {[t['function']['name'] for t in OPENAI_TOOLS]}")

openapi_spec = openai_tools_to_openapi(OPENAI_TOOLS, title="Zava Resolution Desk Tools")
OPENAPI_OUT = WORK / "zava_tools_openapi.json"
OPENAPI_OUT.write_text(json.dumps(openapi_spec, indent=2), encoding="utf-8")

with open(OPENAPI_OUT, "rb") as fh:
    spec_file = model_client.files.create(file=(OPENAPI_OUT.name, fh), purpose="user_data")

for _ in range(20):
    spec_file = model_client.files.retrieve(file_id=spec_file.id)
    if spec_file.status in ("processed", "error"):
        break
    time.sleep(2)
if spec_file.status != "processed":
    raise RuntimeError(f"Tool spec {spec_file.id} ended in status {spec_file.status}")
print(f"Uploaded: {spec_file.id} (status={spec_file.status})")

## 4. Generate synthetic training data

The Foundry datagen service uses the teacher (`gpt-4.1`) to write realistic user prompts plus the correct tool invocations for each. ~50 examples takes 2–5 minutes.

In [ ]:
DATAFILES = [WORK / f"zava-tools-sft-{batch}_dg.jsonl" for batch in range(1, 4)]
if all(path.exists() for path in DATAFILES):
    if RESUME_DATAGEN_JOB_IDS:
        resumed_jobs = [
            project.beta.datasets.get_generation_job(job_id=job_id)
            for job_id in RESUME_DATAGEN_JOB_IDS
        ]
        incomplete = [job.id for job in resumed_jobs if not str(job.status).endswith("SUCCEEDED")]
        if incomplete:
            raise RuntimeError(f"Cached data-generation jobs are not succeeded: {incomplete}")
        RUN_STATE["data_generation_job_ids"] = RESUME_DATAGEN_JOB_IDS
        RUN_STATE["data_generation_statuses"] = [str(job.status) for job in resumed_jobs]
        print("Reusing three completed, service-validated data-generation outputs.")
    else:
        RUN_STATE["data_generation_statuses"] = ["cached"] * len(DATAFILES)
        print("Reusing three cached data-generation outputs.")
else:
    DATAFILES = [
        submit_foundry_datagen(
            file_id=spec_file.id,
            teacher=TEACHER_MODEL,
            max_samples=60,
            output_name=f"zava-tools-sft-{batch}",
        )
        for batch in range(1, 4)
    ]

data = []
seen = set()
for datafile in DATAFILES:
    with open(datafile, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            key = json.dumps(row, sort_keys=True)
            if key not in seen:
                seen.add(key)
                data.append(row)
if len(data) < 20:
    raise RuntimeError(f"Data generation produced only {len(data)} unique rows; at least 20 are required")
RUN_STATE["stages"]["data_generation"] = "succeeded"
RUN_STATE["generated_rows"] = len(data)
print()
print(f"Unique rows: {len(data)}")
print("First row preview:")
print(json.dumps(data[0]["messages"][:2], indent=2)[:500])

## 5. Split into train / val / test

80% train, 10% val, 10% test. The test split is held out from training so we can measure lift fairly.

In [ ]:
rng = random.Random(42)
indices = list(range(len(data))); rng.shuffle(indices)
n_train = int(0.8 * len(indices)); n_val = int(0.1 * len(indices))
train_idx = indices[:n_train]; val_idx = indices[n_train:n_train+n_val]; test_idx = indices[n_train+n_val:]

TRAIN_PATH = WORK / "train.jsonl"
VAL_PATH   = WORK / "val.jsonl"
TEST_PATH  = WORK / "test.jsonl"
for path, idxs in [(TRAIN_PATH, train_idx), (VAL_PATH, val_idx), (TEST_PATH, test_idx)]:
    with path.open("w", encoding="utf-8") as f:
        for i in idxs: f.write(json.dumps(data[i]) + "\n")

print(f"  train: {len(train_idx)} rows")
print(f"  val:   {len(val_idx)} rows")
print(f"  test:  {len(test_idx)} rows")


## 6. Baseline evaluation via the Foundry evals SDK

We use `azure-ai-evaluation.evaluate()` as the driver. Built-in evaluators don't cover tool-call structural matching, so we provide a small custom evaluator function that the SDK runs across all test rows.

In [ ]:
def tool_call_score(ref_calls, out_calls):
    """Structural match. 10 = exact match, 8 = same names different args,
    2-8 = partial name overlap, 1 = no overlap."""
    if not ref_calls: return 10 if not out_calls else 5
    if not out_calls: return 1
    def _name(c):
        if isinstance(c, dict): return ((c.get("function") or {}).get("name")) or c.get("name")
        fn = getattr(c, "function", None)
        return getattr(fn, "name", None) if fn else getattr(c, "name", None)
    def _args(c):
        if isinstance(c, dict):
            raw = ((c.get("function") or {}).get("arguments")) or c.get("arguments")
        else:
            fn = getattr(c, "function", None)
            raw = getattr(fn, "arguments", None) if fn else getattr(c, "arguments", None)
        if isinstance(raw, str):
            try: return json.loads(raw)
            except Exception: return {"_raw": raw}
        return raw or {}
    ref_names = [_name(c) for c in ref_calls]
    out_names = [_name(c) for c in out_calls]
    ref_set, out_set = set(ref_names), set(out_names)
    overlap = ref_set & out_set
    if not overlap: return 1
    if ref_set != out_set:
        ratio = len(overlap) / len(ref_set | out_set)
        return max(2, int(round(2 + ratio * 6)))
    ref_args = {_name(c): _args(c) for c in ref_calls}
    out_args = {_name(c): _args(c) for c in out_calls}
    return 10 if all(ref_args[n] == out_args.get(n) for n in ref_names) else 8


def tool_call_evaluator(*, response, ground_truth, **kwargs):
    """azure-ai-evaluation-compatible evaluator. Called per-row with the
    target's output (response) and the reference (ground_truth)."""
    ref = json.loads(ground_truth) if isinstance(ground_truth, str) else (ground_truth or [])
    out = json.loads(response) if isinstance(response, str) else (response or [])
    score = tool_call_score(ref, out)
    return {"tool_match_score": score, "tool_match_pass": 1.0 if score >= 8 else 0.0}


def make_target(model_name):
    """Returns a retrying callable the evaluate() SDK uses row-by-row."""
    def _target(*, messages, **kwargs):
        conversation = json.loads(messages) if isinstance(messages, str) else messages
        for attempt in range(1, 6):
            try:
                resp = model_client.chat.completions.create(
                    model=model_name,
                    messages=conversation,
                    tools=OPENAI_TOOLS, temperature=0.0, max_completion_tokens=2048,
                )
                tool_calls = resp.choices[0].message.tool_calls or []
                return {"response": json.dumps([
                    {"function": {"name": tc.function.name, "arguments": tc.function.arguments}} for tc in tool_calls
                ])}
            except Exception:
                if attempt == 5:
                    raise
                time.sleep(min(60, 5 * (2 ** (attempt - 1))))
    return _target


# Build one evaluation row for every assistant tool-call turn, preserving its conversation context.
EVAL_DATA_PATH = WORK / "eval_data.jsonl"
eval_rows = 0
with open(TEST_PATH, encoding="utf-8") as fin, open(EVAL_DATA_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        messages = json.loads(line)["messages"]
        for index, message in enumerate(messages):
            ref_tcs = message.get("tool_calls") if message.get("role") == "assistant" else None
            if not ref_tcs:
                continue
            ground_truth = [
                {"function": {
                    "name": (tool_call.get("function") or {}).get("name"),
                    "arguments": (tool_call.get("function") or {}).get("arguments"),
                }}
                for tool_call in ref_tcs
            ]
            fout.write(json.dumps({
                "messages": json.dumps(messages[:index]),
                "ground_truth": json.dumps(ground_truth),
            }) + "\n")
            eval_rows += 1
if not eval_rows:
    raise RuntimeError("The held-out split contained no assistant tool-call turns")
RUN_STATE["evaluation_rows"] = eval_rows
print(f"Eval data: {EVAL_DATA_PATH.name} ({eval_rows} tool-call turns)")

from azure.ai.evaluation import evaluate
print(f"\nBaseline ({STUDENT_MODEL}) evaluation...")
baseline_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(STUDENT_MODEL),
    evaluators={"tool_match": tool_call_evaluator},
    output_path=str(WORK / "baseline_eval_results.json"),
)
baseline_combined = baseline_results["metrics"].get("tool_match.tool_match_score", 0)
baseline_pass = baseline_results["metrics"].get("tool_match.tool_match_pass", 0) * 100
RUN_STATE["baseline"] = {"combined": baseline_combined, "pass_rate": baseline_pass}
RUN_STATE["stages"]["baseline_evaluation"] = "succeeded"
print(f"  Baseline: combined={baseline_combined:.2f}/10  pass_rate={baseline_pass:.1f}%")


## 7. Submit the fine-tuning job

Winning hyperparameters from prior experiments: **`gpt-4.1-mini`, 3 epochs, learning-rate multiplier 1.0**. For your own tools, sweep a few combinations and pick the winner the same way.

In [ ]:
if RESUME_FT_JOB_ID:
    print(f"Reusing fine-tuning job {RESUME_FT_JOB_ID}...")
    ft_job = model_client.fine_tuning.jobs.retrieve(RESUME_FT_JOB_ID)
    train_file = model_client.files.retrieve(file_id=ft_job.training_file)
    val_file = model_client.files.retrieve(file_id=ft_job.validation_file)
else:
    print("Uploading train + val files...")
    with open(TRAIN_PATH, "rb") as fh:
        train_file = model_client.files.create(file=(TRAIN_PATH.name, fh), purpose="fine-tune")
    with open(VAL_PATH, "rb") as fh:
        val_file = model_client.files.create(file=(VAL_PATH.name, fh), purpose="fine-tune")

    for uploaded in (train_file, val_file):
        for _ in range(90):
            uploaded = model_client.files.retrieve(file_id=uploaded.id)
            if uploaded.status in ("processed", "error"):
                break
            time.sleep(2)
        if uploaded.status != "processed":
            raise RuntimeError(f"File {uploaded.id} ended in status {uploaded.status}")
        print(f"  {uploaded.id} status={uploaded.status}")

    print(f"\nSubmitting {TRAINING_TYPE} fine-tuning job...")
    ft_job = model_client.fine_tuning.jobs.create(
        model=STUDENT_MODEL,
        training_file=train_file.id,
        validation_file=val_file.id,
        method={"type": "supervised"},
        hyperparameters={"n_epochs": 3, "learning_rate_multiplier": 1.0},
        suffix="zava-tools-demo",
        extra_body={"trainingType": TRAINING_TYPE},
    )
RUN_STATE["training_file_id"] = train_file.id
RUN_STATE["validation_file_id"] = val_file.id
RUN_STATE["fine_tuning_job_id"] = ft_job.id
RUN_STATE["stages"]["fine_tuning_submission"] = "succeeded"
print(f"  Job: {ft_job.id}  status={ft_job.status}")


## 8. Monitor training

Poll every 30 seconds and print step / loss as training progresses.

In [ ]:
print(f"Monitoring job {ft_job.id}...\n")
last_seen_step = -1
while True:
    job = model_client.fine_tuning.jobs.retrieve(ft_job.id)
    events = list(model_client.fine_tuning.jobs.list_events(fine_tuning_job_id=ft_job.id, limit=10))
    for e in reversed(events):
        msg = e.message or ""
        if "Step " in msg and ":" in msg:
            try:
                step = int(msg.split("Step ")[1].split(":")[0])
                if step > last_seen_step:
                    print(f"  {time.strftime('%H:%M:%S')}  {msg[:90]}")
                    last_seen_step = step
            except (ValueError, IndexError):
                pass
    if job.status in ("succeeded", "failed", "cancelled"):
        print(f"\nFinal status: {job.status}")
        RUN_STATE["fine_tuning_status"] = job.status
        if job.status != "succeeded":
            message = job.error.message if job.error else "(none)"
            print(f"Error: {message}")
            raise RuntimeError(f"Fine-tuning {job.status}: {message}")
        FT_MODEL_ID = job.fine_tuned_model
        RUN_STATE["fine_tuned_model_id"] = FT_MODEL_ID
        RUN_STATE["stages"]["fine_tuning"] = "succeeded"
        print(f"Fine-tuned model: {FT_MODEL_ID}")
        break
    time.sleep(30)


## 9. Validate the fine-tuned deployment

`azure-ai-projects` 2.x exposes stable project deployment lookup but not deployment creation. To avoid raw ARM URLs and account-level configuration, set `AZURE_FINE_TUNED_DEPLOYMENT` to a deployment already created in this project for the completed fine-tuned model. The notebook validates the deployment through `project.deployments` and waits until it is inferenceable.

In [ ]:
deployment = project.deployments.get(DEPLOY_NAME)
print(f"Project deployment: {deployment.name} -> {deployment.model_name}")
if deployment.model_name != FT_MODEL_ID:
    raise RuntimeError(
        f"Deployment {DEPLOY_NAME!r} targets {deployment.model_name!r}, "
        f"not completed fine-tuned model {FT_MODEL_ID!r}"
    )

print("Waiting for deployment to become inferenceable...")
deployment_ready = False
for attempt in range(40):
    try:
        model_client.chat.completions.create(
            model=DEPLOY_NAME,
            messages=[{"role": "user", "content": "hi"}],
            max_completion_tokens=5,
        )
        deployment_ready = True
        print(f"  Ready after {attempt * 30}s")
        break
    except Exception:
        if attempt == 39:
            raise
        time.sleep(30)
        print(".", end="", flush=True)
if not deployment_ready:
    raise TimeoutError("Fine-tuned deployment did not become inferenceable within 20 minutes")
RUN_STATE["deployment_name"] = DEPLOY_NAME
RUN_STATE["deployment_model"] = deployment.model_name
RUN_STATE["stages"]["deployment_validation"] = "succeeded"

## 10. Evaluate the fine-tuned model and compare

Same test set, same evaluator, side-by-side comparison.

In [ ]:
print(f"Fine-tuned ({STUDENT_MODEL}) evaluation...")
ft_results = evaluate(
    data=str(EVAL_DATA_PATH),
    target=make_target(DEPLOY_NAME),
    evaluators={"tool_match": tool_call_evaluator},
    output_path=str(WORK / "ft_eval_results.json"),
)
ft_combined = ft_results["metrics"].get("tool_match.tool_match_score", 0)
ft_pass = ft_results["metrics"].get("tool_match.tool_match_pass", 0) * 100

print()
print("-" * 60)
print(f"  {'Model':<35}  {'Combined':>10}  {'Pass Rate':>10}")
print(f"  {'-'*35}  {'-'*10}  {'-'*10}")
print(f"  Baseline ({STUDENT_MODEL}){' '*(35-19-len(STUDENT_MODEL))}  {baseline_combined:>10.2f}  {baseline_pass:>9.1f}%")
print(f"  Fine-tuned ({STUDENT_MODEL}){' '*(35-21-len(STUDENT_MODEL))}  {ft_combined:>10.2f}  {ft_pass:>9.1f}%")

lift = (ft_combined - baseline_combined) / baseline_combined * 100 if baseline_combined > 0 else 0
print(f"\n  Lift:  {lift:+.1f}%")
if lift >= 5:
    print(f"  Fine-tuning improved combined score by {lift:+.1f}% -- ship the FT model.")
else:
    print(f"  Lift below 5% threshold. Consider: more training data, different HPs, or a larger student model.")

RUN_STATE["fine_tuned_evaluation"] = {
    "combined": ft_combined,
    "pass_rate": ft_pass,
    "lift_percent": lift,
}
RUN_STATE["stages"]["fine_tuned_evaluation"] = "succeeded"
RUN_STATE["validated_at_utc"] = datetime.now(timezone.utc).isoformat()
RUN_STATE["package_versions"] = {
    name: version(name)
    for name in ("azure-ai-projects", "azure-ai-evaluation", "azure-identity", "openai")
}
(WORK / "run_summary.json").write_text(json.dumps(RUN_STATE, indent=2), encoding="utf-8")
print(f"Run summary: {WORK / 'run_summary.json'}")

## Cleanup

Delete uploaded files through the project-derived model client when they are no longer needed:

```python
model_client.files.delete(spec_file.id)
model_client.files.delete(train_file.id)
model_client.files.delete(val_file.id)
```

Delete the model deployment through the Foundry portal or your approved resource-management workflow; `azure-ai-projects` 2.x currently exposes deployment lookup, not deployment creation or deletion.

## Bring your own tools

Replace `fixtures/zava_tools_openai.json` with your own tool catalog in the OpenAI chat-completions tool format. The notebook converts it to OpenAPI 3.0 automatically.

## Dependencies

```bash
pip install "azure-ai-projects>=2.7,<3" "azure-identity>=1.26,<2" "azure-ai-evaluation>=1.18.7,<2" jupyter
```